In [1]:
import torch
import torch.nn as nn
import numpy as np
import plotly.graph_objects as go
import random
import copy
from tqdm import tqdm

class MLP(nn.Module):
  def __init__(self, input_size):
    super().__init__()
    self.layers = nn.Sequential(
        nn.Linear(input_size, 128),
        nn.ReLU(),
        nn.Linear(128, 256),
        nn.ReLU(),
        nn.Linear(256, 256),
        nn.ReLU(),
        nn.Linear(256, 64),
        nn.ReLU(),
        nn.Linear(64, 1)
    )

  def forward(self, x):
    return self.layers(x)

In [2]:
X = torch.tensor(np.load("X_data.npy"), dtype=torch.float64)
y = torch.tensor(np.load("y_data.npy"), dtype=torch.float64)

model = MLP(input_size=45)
state_dict = torch.load("car_regression.pth", map_location=torch.device("cpu"))
model.load_state_dict(state_dict)

<All keys matched successfully>

In [3]:
θ = nn.utils.parameters_to_vector(model.parameters()).cpu().detach().numpy()
d = np.array([random.gauss(0,1) for _ in range(θ.size)])
η = np.array([random.gauss(0,1) for _ in range(θ.size)])

In [ ]:
index_pointer = 0
for param in model.parameters():
  if param.dim() == 2:
    for filter in param:
      filter_norm = np.linalg.norm(filter.cpu().detach().numpy())
      d[index_pointer:index_pointer + filter.size()[0]] *= \
        filter_norm/np.linalg.norm(d[index_pointer:index_pointer + filter.size()[0]])
      η[index_pointer:index_pointer + filter.size()[0]] *= \
        filter_norm/np.linalg.norm(η[index_pointer:index_pointer + filter.size()[0]])
      index_pointer += filter.size()[0]
  elif param.dim() == 1:
    filter = param
    filter_norm = np.linalg.norm(filter.cpu().detach().numpy())
    d[index_pointer:index_pointer + filter.size()[0]] *= \
      filter_norm/np.linalg.norm(d[index_pointer:index_pointer + filter.size()[0]])
    η[index_pointer:index_pointer + filter.size()[0]] *= \
      filter_norm/np.linalg.norm(η[index_pointer:index_pointer + filter.size()[0]])
    index_pointer += filter.size()[0]
  else:
    raise NotImplementedError(f"Found parameter with dimension {param.dim()}. Parameters must be dimension 1 or 2")

In [33]:
alpha = np.linspace(-1, 1, 100)
beta = np.linspace(-1, 1, 100)
Z = np.zeros((len(alpha), len(beta)))
X.to(torch.device("cuda"))
y.to(torch.device("cuda"))
objective = nn.MSELoss()
model_i = copy.deepcopy(model)
model_i.to(torch.device("cuda"))

MLP(
  (layers): Sequential(
    (0): Linear(in_features=45, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=256, bias=True)
    (3): ReLU()
    (4): Linear(in_features=256, out_features=256, bias=True)
    (5): ReLU()
    (6): Linear(in_features=256, out_features=64, bias=True)
    (7): ReLU()
    (8): Linear(in_features=64, out_features=1, bias=True)
  )
)

In [34]:
with torch.no_grad():
  for i, a in tqdm(enumerate(alpha)):
    for j, b in enumerate(beta):
      θ_new = torch.tensor(θ + (a * d) + (b * η))
      nn.utils.vector_to_parameters(θ_new, model_i.parameters())
      outputs = model_i(X)
      z = objective(outputs, y).item()
      Z[i,j] = z

100it [00:35,  2.82it/s]


In [35]:
fig = go.Figure(
    data = [
        go.Surface(
            x=alpha,
            y=beta,
            z=Z,
            colorscale="Viridis"
        )
    ]
)

fig.show()

In [36]:
fig.write_html("loss_curvature.html")